# 第 5 周第 2 课：系统化数据清洗与可追踪转换

本 Notebook 将基于上一课的数据审计结果：

- 保留不可修改的原始输入；
- 严格转换日期和类别；
- 新增可读标签及业务尺度；
- 记录每一步转换的理由和影响；
- 验证行数、业务键、字段关系和总租赁量守恒。

In [1]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../data/raw/hour.csv")
PROCESSED_PATH = Path("../data/processed/hourly_clean.csv")

assert RAW_PATH.exists(), f"找不到原始文件: {RAW_PATH.resolve()}"

raw = pd.read_csv(RAW_PATH)
df = raw.copy()

In [2]:
baseline = {
    "rows": len(raw),
    "columns": len(raw.columns),
    "duplicate_rows": int(raw.duplicated().sum()),
    "missing_cells": int(raw.isna().sum().sum()),
    "cnt_total": int(raw["cnt"].sum()),
}

baseline

{'rows': 17379,
 'columns': 17,
 'duplicate_rows': 0,
 'missing_cells': 0,
 'cnt_total': 3292679}

In [3]:
raw[["dteday", "hr"]].dtypes

df["dteday"] = pd.to_datetime(
    df["dteday"],
    format="%Y-%m-%d",
    errors="raise",
)

df["timestamp"] = (
    df["dteday"]
    + pd.to_timedelta(df["hr"], unit="h")
)

print(df[["dteday", "timestamp"]].dtypes)
print(df[["dteday", "hr", "timestamp"]].head())

dteday       datetime64[ns]
timestamp    datetime64[ns]
dtype: object
      dteday  hr           timestamp
0 2011-01-01   0 2011-01-01 00:00:00
1 2011-01-01   1 2011-01-01 01:00:00
2 2011-01-01   2 2011-01-01 02:00:00
3 2011-01-01   3 2011-01-01 03:00:00
4 2011-01-01   4 2011-01-01 04:00:00


In [4]:
assert df["dteday"].notna().all()
assert df["timestamp"].notna().all()
assert df["timestamp"].is_unique
assert df["timestamp"].is_monotonic_increasing
assert len(df) == baseline["rows"]
assert int(df["cnt"].sum()) == baseline["cnt_total"]

print("日期与时间戳转换验证通过")

日期与时间戳转换验证通过


In [5]:
df[["timestamp"]].agg(["min", "max"])

,timestamp
min,2011-01-01 00:00:00
max,2012-12-31 23:00:00


In [6]:
SEASON_LABELS = {
    1: "spring",
    2: "summer",
    3: "fall",
    4: "winter",
}

WEATHER_LABELS = {
    1: "clear_or_partly_cloudy",
    2: "mist_or_cloudy",
    3: "light_rain_or_snow",
    4: "heavy_rain_snow_or_fog",
}

WEEKDAY_LABELS = {
    0: "sunday",
    1: "monday",
    2: "tuesday",
    3: "wednesday",
    4: "thursday",
    5: "friday",
    6: "saturday",
}

In [7]:
df["season_label"] = df["season"].map(SEASON_LABELS)
df["weather_label"] = df["weathersit"].map(WEATHER_LABELS)
df["weekday_label"] = df["weekday"].map(WEEKDAY_LABELS)

In [8]:
label_columns = [
    "season_label",
    "weather_label",
    "weekday_label",
]

print(df[label_columns].isna().sum())

assert df[label_columns].notna().all().all()

print("类别编码全部成功映射")

season_label     0
weather_label    0
weekday_label    0
dtype: int64
类别编码全部成功映射


In [9]:
for column in label_columns:
    df[column] = df[column].astype("category")

df[
    [
        "season",
        "season_label",
        "weekday",
        "weekday_label",
        "weathersit",
        "weather_label",
    ]
].head()

df[label_columns].dtypes

for column in label_columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False))


--- season_label ---
season_label
fall      4496
summer    4409
spring    4242
winter    4232
Name: count, dtype: int64

--- weather_label ---
weather_label
clear_or_partly_cloudy    11413
mist_or_cloudy             4544
light_rain_or_snow         1419
heavy_rain_snow_or_fog        3
Name: count, dtype: int64

--- weekday_label ---
weekday_label
saturday     2512
sunday       2502
friday       2487
monday       2479
wednesday    2475
thursday     2471
tuesday      2453
Name: count, dtype: int64


In [10]:
print(df[label_columns].isna().sum())
print()
print(df[label_columns].dtypes)
print()
print(df["weather_label"].value_counts(dropna=False))

season_label     0
weather_label    0
weekday_label    0
dtype: int64

season_label     category
weather_label    category
weekday_label    category
dtype: object

weather_label
clear_or_partly_cloudy    11413
mist_or_cloudy             4544
light_rain_or_snow         1419
heavy_rain_snow_or_fog        3
Name: count, dtype: int64


In [11]:
df[["temp", "atemp", "hum", "windspeed"]].head()

df = df.assign(
    temp_c=df["temp"] * 41,
    feels_like_c=df["atemp"] * 50,
    humidity_pct=df["hum"] * 100,
    windspeed_scaled=df["windspeed"] * 67,
)

In [12]:
assert df["temp_c"].between(0, 41).all()
assert df["feels_like_c"].between(0, 50).all()
assert df["humidity_pct"].between(0, 100).all()
assert df["windspeed_scaled"].between(0, 67).all()

assert len(df) == baseline["rows"]
assert int(df["cnt"].sum()) == baseline["cnt_total"]

print("业务尺度恢复及守恒验证通过")

业务尺度恢复及守恒验证通过


In [13]:
df[
    [
        "temp_c",
        "feels_like_c",
        "humidity_pct",
        "windspeed_scaled",
    ]
].agg(["min", "max"])

,temp_c,feels_like_c,humidity_pct,windspeed_scaled
min,0.82,0.0,0.0,0.0000
max,41.00,50.0,100.0,56.9969


In [14]:
df[
    [
        "temp",
        "temp_c",
        "hum",
        "humidity_pct",
        "windspeed",
        "windspeed_scaled",
    ]
].head()

,temp,temp_c,hum,humidity_pct,windspeed,windspeed_scaled
0,0.24,9.84,0.81,81.0,0.0,0.0
1,0.22,9.02,0.80,80.0,0.0,0.0
2,0.22,9.02,0.80,80.0,0.0,0.0
3,0.24,9.84,0.75,75.0,0.0,0.0
4,0.24,9.84,0.75,75.0,0.0,0.0


In [15]:
df[label_columns].dtypes

season_label     category
weather_label    category
weekday_label    category
dtype: object

In [16]:
assert df.isna().sum().sum() == 0

In [17]:
full_hours = pd.date_range(
    start=df["timestamp"].min(),
    end=df["timestamp"].max(),
    freq="h",
)

missing_hours = full_hours.difference(df["timestamp"])

print("理论小时数:", len(full_hours))
print("实际记录数:", len(df))
print("缺失小时数:", len(missing_hours))
print()
print("前 10 个缺失小时:")
print(missing_hours[:10])

理论小时数: 17544
实际记录数: 17379
缺失小时数: 165

前 10 个缺失小时:


DatetimeIndex(['2011-01-02 05:00:00', '2011-01-03 02:00:00',
               '2011-01-03 03:00:00', '2011-01-04 03:00:00',
               '2011-01-05 03:00:00', '2011-01-06 03:00:00',
               '2011-01-07 03:00:00', '2011-01-11 03:00:00',
               '2011-01-11 04:00:00', '2011-01-12 03:00:00'],
              dtype='datetime64[ns]', freq=None)


In [18]:
assert len(full_hours) == 17_544
assert len(df) == 17_379
assert len(missing_hours) == 165

In [19]:
missing_hour_counts = (
    pd.Series(missing_hours.hour, name="hour")
    .value_counts()
    .sort_index()
)

missing_hour_counts

hour
0      5
1      7
2     16
3     34
4     34
5     14
6      6
7      4
8      4
9      4
10     4
11     4
12     3
13     2
14     2
15     2
16     1
17     1
18     3
19     3
20     3
21     3
22     3
23     3
Name: count, dtype: int64

In [20]:
missing_dates = pd.Series(
    missing_hours.normalize(),
    name="date",
)

missing_by_date = missing_dates.value_counts().sort_index()

print("涉及日期数:", len(missing_by_date))
print("单日最大缺失小时数:", missing_by_date.max())
print()
print(missing_by_date.head(10))

涉及日期数: 76
单日最大缺失小时数: 23



date
2011-01-02     1
2011-01-03     2
2011-01-04     1
2011-01-05     1
2011-01-06     1
2011-01-07     1
2011-01-11     2
2011-01-12     2
2011-01-14     1
2011-01-18    12
Name: count, dtype: int64


In [21]:
demand_timeline = (
    df.set_index("timestamp")[["casual", "registered", "cnt"]]
    .reindex(full_hours)
    .rename_axis("timestamp")
)

In [22]:
missing_record = demand_timeline["cnt"].isna()

demand_timeline["record_was_missing"] = missing_record

print(demand_timeline["record_was_missing"].value_counts())

record_was_missing
False    17379
True       165
Name: count, dtype: int64


In [23]:
count_columns = ["casual", "registered", "cnt"]

demand_timeline.loc[
    missing_record,
    count_columns,
] = 0

demand_timeline[count_columns] = (
    demand_timeline[count_columns].astype("int64")
)

In [24]:
assert len(demand_timeline) == 17_544
assert demand_timeline["record_was_missing"].sum() == 165
assert demand_timeline[count_columns].notna().all().all()
assert demand_timeline[count_columns].ge(0).all().all()
assert demand_timeline["cnt"].eq(
    demand_timeline["casual"] + demand_timeline["registered"]
).all()
assert int(demand_timeline["cnt"].sum()) == baseline["cnt_total"]

print("完整需求时间轴验证通过")

完整需求时间轴验证通过


In [25]:
duplicate_rows = int(df.duplicated().sum())

duplicate_business_keys = int(
    df.duplicated(
        subset=["dteday", "hr"],
        keep=False,
    ).sum()
)

print("完整重复行:", duplicate_rows)
print("业务键重复行:", duplicate_business_keys)

assert duplicate_rows == 0
assert duplicate_business_keys == 0

完整重复行: 0
业务键重复行: 0


In [26]:
sample = pd.DataFrame(
    {
        "key": ["A", "A", "B"],
        "value": [10, 20, 30],
    }
)

sample["duplicated_default"] = sample.duplicated(subset=["key"])
sample["duplicated_keep_false"] = sample.duplicated(
    subset=["key"],
    keep=False,
)

sample

,key,value,duplicated_default,duplicated_keep_false
0,A,10,False,True
1,A,20,True,True
2,B,30,False,False


In [27]:
q1 = df["cnt"].quantile(0.25)
q3 = df["cnt"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("IQR 上界:", upper_bound)

Q1: 40.0
Q3: 281.0
IQR: 241.0
IQR 上界: 642.5


In [28]:
df["cnt_iqr_high"] = df["cnt"] > upper_bound

high_count = int(df["cnt_iqr_high"].sum())
high_rate = df["cnt_iqr_high"].mean()

print("高值候选数:", high_count)
print(f"高值候选比例: {high_rate:.2%}")

高值候选数: 505
高值候选比例: 2.91%


In [29]:
df.loc[
    df["cnt_iqr_high"],
    [
        "timestamp",
        "season_label",
        "weekday_label",
        "workingday",
        "hr",
        "weather_label",
        "cnt",
    ],
].sort_values("cnt", ascending=False).head(10)

,timestamp,season_label,weekday_label,workingday,hr,weather_label,cnt
14773,2012-09-12 18:00:00,fall,wednesday,1,18,clear_or_partly_cloudy,977
14964,2012-09-20 17:00:00,fall,thursday,1,17,clear_or_partly_cloudy,976
14748,2012-09-11 17:00:00,fall,tuesday,1,17,clear_or_partly_cloudy,970
14725,2012-09-10 18:00:00,fall,monday,1,18,clear_or_partly_cloudy,968
15084,2012-09-25 17:00:00,winter,tuesday,1,17,clear_or_partly_cloudy,967
15780,2012-10-24 17:00:00,winter,wednesday,1,17,clear_or_partly_cloudy,963
10622,2012-03-23 17:00:00,summer,friday,1,17,mist_or_cloudy,957
15108,2012-09-26 17:00:00,winter,wednesday,1,17,clear_or_partly_cloudy,953
15444,2012-10-10 17:00:00,winter,wednesday,1,17,clear_or_partly_cloudy,948
15588,2012-10-16 17:00:00,winter,tuesday,1,17,clear_or_partly_cloudy,943


In [30]:
df.loc[
    df["cnt_iqr_high"],
    "hr",
].value_counts().sort_values(ascending=False)

hr
17    153
18    129
8     127
13     23
12     17
14     15
19     14
15     13
16     12
11      2
Name: count, dtype: int64

In [31]:
assert len(df) == baseline["rows"]
assert int(df["cnt"].sum()) == baseline["cnt_total"]
assert df["cnt"].eq(df["casual"] + df["registered"]).all()

print("重复与高值处理验证通过")

重复与高值处理验证通过


## 可复现的清洗函数

下面把类别映射和核心转换封装成函数，使清洗过程不依赖 Notebook 单元格的隐藏状态。

In [32]:
def map_required(
    series: pd.Series,
    mapping: dict[int, str],
    field_name: str,
) -> pd.Series:
    unknown_codes = sorted(
        set(series.dropna().unique()) - set(mapping)
    )

    if unknown_codes:
        raise ValueError(
            f"{field_name} 存在未知编码: {unknown_codes}"
        )

    return series.map(mapping)

In [33]:
def clean_hourly_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    cleaned = raw_df.copy()

    cleaned["dteday"] = pd.to_datetime(
        cleaned["dteday"],
        format="%Y-%m-%d",
        errors="raise",
    )

    cleaned["timestamp"] = (
        cleaned["dteday"]
        + pd.to_timedelta(cleaned["hr"], unit="h")
    )

    cleaned["season_label"] = map_required(
        cleaned["season"],
        SEASON_LABELS,
        "season",
    )
    cleaned["weather_label"] = map_required(
        cleaned["weathersit"],
        WEATHER_LABELS,
        "weathersit",
    )
    cleaned["weekday_label"] = map_required(
        cleaned["weekday"],
        WEEKDAY_LABELS,
        "weekday",
    )

    current_label_columns = [
        "season_label",
        "weather_label",
        "weekday_label",
    ]

    for column in current_label_columns:
        cleaned[column] = cleaned[column].astype("category")

    cleaned = cleaned.assign(
        temp_c=cleaned["temp"] * 41,
        feels_like_c=cleaned["atemp"] * 50,
        humidity_pct=cleaned["hum"] * 100,
        windspeed_scaled=cleaned["windspeed"] * 67,
    )

    q1 = cleaned["cnt"].quantile(0.25)
    q3 = cleaned["cnt"].quantile(0.75)
    upper_bound = q3 + 1.5 * (q3 - q1)

    cleaned["cnt_iqr_high"] = cleaned["cnt"] > upper_bound

    return cleaned

In [34]:
raw_again = pd.read_csv(RAW_PATH)

before_columns = raw_again.columns.tolist()
before_rows = len(raw_again)
before_cnt_total = int(raw_again["cnt"].sum())

cleaned = clean_hourly_data(raw_again)

In [35]:
assert raw_again.columns.tolist() == before_columns
assert len(raw_again) == before_rows
assert int(raw_again["cnt"].sum()) == before_cnt_total

assert "timestamp" not in raw_again.columns
assert "season_label" not in raw_again.columns

print("输入 DataFrame 未被修改")

输入 DataFrame 未被修改


In [36]:
cleaned_label_columns = [
    "season_label",
    "weather_label",
    "weekday_label",
]

assert cleaned is not raw_again
assert len(cleaned) == baseline["rows"]

assert cleaned["dteday"].notna().all()
assert cleaned["timestamp"].notna().all()
assert cleaned["timestamp"].is_unique
assert cleaned["timestamp"].is_monotonic_increasing

assert cleaned[cleaned_label_columns].notna().all().all()
assert all(
    str(cleaned[column].dtype) == "category"
    for column in cleaned_label_columns
)

assert cleaned["cnt"].eq(
    cleaned["casual"] + cleaned["registered"]
).all()

assert int(cleaned["cnt"].sum()) == baseline["cnt_total"]
assert int(cleaned["cnt_iqr_high"].sum()) == 505

print("清洗函数输出契约全部通过")

清洗函数输出契约全部通过


In [37]:
new_columns = [
    column
    for column in cleaned.columns
    if column not in raw_again.columns
]

print("新增字段数:", len(new_columns))
print("新增字段:", new_columns)
print("输出形状:", cleaned.shape)

新增字段数: 9
新增字段: ['timestamp', 'season_label', 'weather_label', 'weekday_label', 'temp_c', 'feels_like_c', 'humidity_pct', 'windspeed_scaled', 'cnt_iqr_high']
输出形状: (17379, 26)


In [38]:
broken = raw_again.head(3).copy()
broken.loc[broken.index[0], "season"] = 5

try:
    clean_hourly_data(broken)
except ValueError as error:
    print(type(error).__name__)
    print(error)

ValueError
season 存在未知编码: [5]


In [39]:
transformation_log = pd.DataFrame(
    [
        {
            "step": "parse_date_and_build_timestamp",
            "reason": "严格解析日期并建立小时级业务时间键",
            "rows_before": len(raw_again),
            "rows_after": len(cleaned),
            "affected_rows": len(cleaned),
        },
        {
            "step": "add_category_labels",
            "reason": "依据官方数据字典添加可读类别标签",
            "rows_before": len(cleaned),
            "rows_after": len(cleaned),
            "affected_rows": len(cleaned),
        },
        {
            "step": "restore_business_scales",
            "reason": "依据官方恢复系数新增业务尺度字段",
            "rows_before": len(cleaned),
            "rows_after": len(cleaned),
            "affected_rows": len(cleaned),
        },
        {
            "step": "flag_iqr_high_demand",
            "reason": "标记高需求候选但不删除真实记录",
            "rows_before": len(cleaned),
            "rows_after": len(cleaned),
            "affected_rows": int(cleaned["cnt_iqr_high"].sum()),
        },
    ]
)

transformation_log

,step,reason,rows_before,rows_after,affected_rows
0,parse_date_and_build_timestamp,严格解析日期并建立小时级业务时间键,17379,17379,17379
1,add_category_labels,依据官方数据字典添加可读类别标签,17379,17379,17379
2,restore_business_scales,依据官方恢复系数新增业务尺度字段,17379,17379,17379
3,flag_iqr_high_demand,标记高需求候选但不删除真实记录,17379,17379,505


In [40]:
assert transformation_log["rows_before"].eq(
    transformation_log["rows_after"]
).all()

print("所有转换均未改变主清洗表行数")

所有转换均未改变主清洗表行数


In [41]:
PROCESSED_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

In [42]:
cleaned.to_csv(
    PROCESSED_PATH,
    index=False,
)

print("输出路径:", PROCESSED_PATH.resolve())
print("文件大小:", PROCESSED_PATH.stat().st_size, "bytes")

输出路径: C:\Users\gjnzsu\Documents\ai-python-learning-pathway\projects\week-05-bike-sharing\data\processed\hourly_clean.csv
文件大小: 2902772 bytes


In [43]:
reloaded = pd.read_csv(
    PROCESSED_PATH,
    parse_dates=["dteday", "timestamp"],
)

In [44]:
assert reloaded.shape == cleaned.shape
assert len(reloaded) == baseline["rows"]

assert reloaded["dteday"].notna().all()
assert reloaded["timestamp"].notna().all()
assert reloaded["timestamp"].is_unique
assert reloaded["timestamp"].is_monotonic_increasing

assert reloaded["cnt"].eq(
    reloaded["casual"] + reloaded["registered"]
).all()

assert int(reloaded["cnt"].sum()) == baseline["cnt_total"]
assert int(reloaded["cnt_iqr_high"].sum()) == 505

assert reloaded[
    ["season_label", "weather_label", "weekday_label"]
].notna().all().all()

print("导出文件回读验证全部通过")

导出文件回读验证全部通过


In [45]:
type_comparison = pd.DataFrame(
    {
        "before_export": cleaned.dtypes.astype(str),
        "after_reload": reloaded.dtypes.astype(str),
    }
)

type_comparison.loc[
    [
        "dteday",
        "timestamp",
        "season_label",
        "weather_label",
        "weekday_label",
        "cnt_iqr_high",
    ]
]

,before_export,after_reload
dteday,datetime64[ns],datetime64[ns]
timestamp,datetime64[ns],datetime64[ns]
season_label,category,object
weather_label,category,object
weekday_label,category,object
cnt_iqr_high,bool,bool


In [46]:
for column in cleaned_label_columns:
    reloaded[column] = reloaded[column].astype("category")

In [47]:
assert all(
    str(reloaded[column].dtype) == "category"
    for column in cleaned_label_columns
)

print("回读后的类别类型已恢复")

回读后的类别类型已恢复
